# Youtube Scrapper

In [16]:
# STEP 1:Importing the necessary libraries
import chromedriver_binary 
from selenium import webdriver
from bs4 import BeautifulSoup
import time

In [17]:
# STEP 2:Setting up the WebDriver
driver = webdriver.Chrome()
driver.get("https://www.youtube.com/c/TechWithTim/videos")

The chromedriver version (147.0.7727.57) detected in PATH at c:\Users\Nbinary\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\chromedriver_binary\chromedriver.exe might not be compatible with the detected chrome version (148.0.7778.167); currently, chromedriver 148.0.7778.167 is recommended for chrome 148.*, so it is advised to delete the driver in PATH and retry


In [18]:
# STEP 3:Scrolling to load more videos
last_height = driver.execute_script("return document.documentElement.scrollHeight")
while True:
    driver.execute_script("window.scrollTo(0, document.documentElement.scrollHeight);")
    time.sleep(2)  # Wait for the page to load
    new_height = driver.execute_script("return document.documentElement.scrollHeight")
    if new_height == last_height:
        break
    last_height = new_height

# Add extra wait to ensure page fully loads
time.sleep(3)

In [19]:
# STEP 4: Parsing the page source with BeautifulSoup
soup = BeautifulSoup(driver.page_source, 'html.parser')

# Try different selectors - modern YouTube uses these
video_elements = soup.find_all('ytd-rich-item-renderer')
print(f"Found {len(video_elements)} ytd-rich-item-renderer elements\n")

if video_elements:
    # Inspect the first element's structure
    first_video = video_elements[0]
    print("First video element structure:")
    print(f"Tag name: {first_video.name}")
    print(f"Direct children: {[child.name for child in first_video.children if hasattr(child, 'name') and child.name][:5]}")
    print(f"\nSearching for links in first element:")
    links = first_video.find_all('a')
    print(f"  - Found {len(links)} <a> tags")
    for j, link in enumerate(links[:3]):
        print(f"    Link {j}: id='{link.get('id')}', class='{link.get('class')}', href='{link.get('href')}'")
    
    print(f"\nSearching for images:")
    imgs = first_video.find_all('img')
    print(f"  - Found {len(imgs)} <img> tags")
    if imgs:
        print(f"    First image: src='{imgs[0].get('src')[:80]}...'")

Found 30 ytd-rich-item-renderer elements

First video element structure:
Tag name: ytd-rich-item-renderer
Direct children: ['div', 'yt-interaction']

Searching for links in first element:
  - Found 2 <a> tags
    Link 0: id='None', class='['ytLockupViewModelContentImage']', href='/watch?v=bzV2EwDyxpk'
    Link 1: id='None', class='['ytLockupMetadataViewModelTitle']', href='/watch?v=bzV2EwDyxpk'

Searching for images:
  - Found 1 <img> tags
    First image: src='https://i.ytimg.com/vi/bzV2EwDyxpk/hqdefault.jpg?sqp=-oaymwEnCNACELwBSFryq4qpAxk...'


In [20]:
# STEP 5: Extracting video details
videos = []

for i, video in enumerate(video_elements):
    try:
        # Find title - it's in the link with class 'ytLockupMetadataViewModelTitle'
        title_link = video.find('a', {'class': 'ytLockupMetadataViewModelTitle'})
        
        if title_link:
            title = title_link.get('aria-label', title_link.text.strip())
            href = title_link.get('href', '')
            url = f"https://www.youtube.com{href}" if href.startswith('/') else href
            
            # Find thumbnail
            img_elem = video.find('img')
            thumbnail = img_elem.get('src', '') if img_elem else ''
            
            videos.append({
                'title': title,
                'url': url,
                'thumbnail': thumbnail
            })
    except Exception as e:
        print(f"Error processing video {i}: {e}")

print(f"✓ Successfully extracted {len(videos)} videos\n")
if videos:
    print("First 3 videos:")
    for v in videos[:3]:
        print(f"  - {v['title']}")
        print(f"    URL: {v['url']}\n")
    print(f"Total videos list: {videos}")
else:
    print("⚠️ No videos extracted!")

✓ Successfully extracted 30 videos

First 3 videos:
  - Connect Claude to ANY Tool | Full Tutorial 12 minutes, 47 seconds
    URL: https://www.youtube.com/watch?v=bzV2EwDyxpk

  - Claude Just Got a Superpower No One's Talking About 15 minutes
    URL: https://www.youtube.com/watch?v=rrylSizvnSg

  - Mistral's New Coding Agent Has a Trick Nobody Expected 16 minutes
    URL: https://www.youtube.com/watch?v=SVz-y-pD6s4

Total videos list: [{'title': 'Connect Claude to ANY Tool | Full Tutorial 12 minutes, 47 seconds', 'url': 'https://www.youtube.com/watch?v=bzV2EwDyxpk', 'thumbnail': 'https://i.ytimg.com/vi/bzV2EwDyxpk/hqdefault.jpg?sqp=-oaymwEnCNACELwBSFryq4qpAxkIARUAAIhCGAHYAQHiAQoIGBACGAY4AUAB&rs=AOn4CLBX7237HoTEvY2lsujegsTrrGUfiQ'}, {'title': "Claude Just Got a Superpower No One's Talking About 15 minutes", 'url': 'https://www.youtube.com/watch?v=rrylSizvnSg', 'thumbnail': 'https://i.ytimg.com/vi/rrylSizvnSg/hqdefault.jpg?sqp=-oaymwEnCNACELwBSFryq4qpAxkIARUAAIhCGAHYAQHiAQoIGBACGAY4AUAB